# 07 — Taxi Daily Gold Fact

## 1. Read Silver Taxi Data

Read the prepared taxi trips from the Silver layer.

In [ ]:
df_taxi = spark.table("dbo.silver_taxi_trips")

print("Rows:", df_taxi.count())

## 2. Filter Valid Taxi Trips

Keep trips within the reporting period and exclude invalid trip durations.

In [ ]:
df_valid_taxi = df_taxi.filter(
    (df_taxi.is_in_reporting_period == True)
    &
    (df_taxi.is_negative_duration == False)
    &
    (df_taxi.is_zero_duration == False)
    &
    (df_taxi.is_long_duration == False)
)

print("Valid Trips:", df_valid_taxi.count())

## 3. Build Hourly Trip Counts

Extract the pickup hour and aggregate taxi trips by date, hour, and pickup zone.

This intermediate DataFrame is used to calculate daily trip counts.

In [ ]:
from pyspark.sql.functions import hour

df_valid_taxi = df_valid_taxi.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [ ]:
df_taxi_hourly = (
    df_valid_taxi
    .groupBy("trip_date", "pickup_hour", "PULocationID")
    .count()
    .withColumnRenamed("count", "trip_count")
)

## 4. Build Daily Trip Counts

Aggregate hourly taxi trips into daily trip counts by pickup zone.

In [ ]:
from pyspark.sql.functions import sum

df_taxi_daily = (
    df_taxi_hourly
    .groupBy("trip_date", "PULocationID")
    .agg(
        sum("trip_count").alias("daily_trips")
    )
)

df_taxi_daily.show(5)

## 5. Calculate Daily Revenue and Average Metrics

Calculate total revenue, average fare, and average trip duration by date and pickup zone.

In [ ]:
from pyspark.sql.functions import sum

df_taxi_revenue = (
    df_valid_taxi
    .groupBy("trip_date", "PULocationID")
    .agg(
        sum("total_amount").alias("total_amount_usd")
    )
)

df_taxi_revenue.show(5)

In [ ]:
from pyspark.sql.functions import avg
df_taxi_averages = (
    df_valid_taxi
    .groupBy("trip_date", "PULocationID")
    .agg(
        avg("fare_amount").alias("avg_fare_usd"),
        avg("trip_duration_minutes").alias("avg_duration_minutes")
    )
)

## 6. Build Daily Taxi Fact

Join daily trip counts, revenue, and average metrics by trip date and pickup zone.

In [ ]:
df_fact_taxi_daily = (
    df_taxi_daily
    .join(
        df_taxi_revenue,
        ["trip_date", "PULocationID"],
        "inner"
    )
    .join(
        df_taxi_averages,
        ["trip_date", "PULocationID"],
        "inner"
    )
)

df_fact_taxi_daily.show(5)

## 7. Validate Fact Join

Check whether the inner joins preserved all daily date-zone records.

In [ ]:
rows_before = df_taxi_daily.count()
rows_after = df_fact_taxi_daily.count()


print("Rows Before:", rows_before)
print("Rows After:", rows_after)
print("Difference:", rows_after - rows_before)

In [ ]:
df_fact_taxi_daily.agg(
    sum("daily_trips").alias("total_trips")
).show()

## 8. Check Negative Payment Amounts

Count taxi trips with negative total amounts before calculating positive revenue.

In [ ]:
negative_amount_count = df_valid_taxi.filter(df_valid_taxi.total_amount < 0).count()

print("Negative Total Amount:", negative_amount_count)

## 9. Calculate Positive Payment Revenue

Select trips with positive total amounts and aggregate revenue by date and pickup zone.

In [ ]:
from pyspark.sql.functions import col

df_paid_taxi = df_valid_taxi.filter(
    (col("payment_type").isin(0,1,2))
    &
    (df_valid_taxi.total_amount > 0)
)

print("Positive Paid Trips:", df_paid_taxi.count())

In [ ]:
df_positive_revenue = (
    df_paid_taxi
    .groupBy("trip_date", "PULocationID")
    .agg(
        sum("total_amount").alias("positive_charged_amount_usd") 
    )
)

df_positive_revenue.show(5)

## 10. Add Positive Revenue to Daily Fact

Join positive payment revenue with daily taxi metrics by date and pickup zone.

In [ ]:
df_fact_taxi_daily_v2  = df_fact_taxi_daily.join(
    df_positive_revenue,
    ["trip_date", "PULocationID"],
    "left"
)

df_fact_taxi_daily_v2.show(5)

In [ ]:
print("Rows:", df_fact_taxi_daily_v2.count())

In [ ]:
null_positive_amount = df_fact_taxi_daily_v2.filter(
    df_fact_taxi_daily_v2.positive_charged_amount_usd.isNull()
).count()

print("NULL Positive Amount:", null_positive_amount)

## 11. Handle Missing Positive Revenue

Replace NULL positive revenue with zero for date-zone groups without qualifying payments.

In [ ]:
df_fact_taxi_daily_v2 = df_fact_taxi_daily_v2.fillna({
    "positive_charged_amount_usd": 0
})

print(
    "NULL Positive Amount:",
    df_fact_taxi_daily_v2.filter(
        col("positive_charged_amount_usd").isNull()
    ).count()
)

## 12. Read ECB Exchange Rates

Read prepared USD/EUR exchange rates from the Silver layer.

In [ ]:
df_fx = spark.table("dbo.silver_ecb_fx")

df_fx.orderBy("rate_date").show(5)

df_fx.printSchema()

## 13. Prepare Historical ECB Exchange Rate

Read historical ECB exchange rates and extract the last available rate before June 2025.

In [ ]:
df_fx_year = (
    spark.read
    .option("header", True)
    .csv("Files/bronze/ecb/fx_usd_2025_raw.csv")
)

df_fx_year.select(
    "TIME_PERIOD",
    "OBS_VALUE"
).filter(
    col("TIME_PERIOD").between("2025-05-26", "2025-06-02")
).orderBy("TIME_PERIOD").show()

In [ ]:
from pyspark.sql.functions import col, to_date

df_fx_may30 = (
    df_fx_year
    .filter(col("TIME_PERIOD") == "2025-05-30")
    .select(
        to_date("TIME_PERIOD").alias("rate_date"),
        col("OBS_VALUE").cast("double").alias("usd_per_eur")
    )
)

df_fx_may30.show()
df_fx_may30.printSchema()

## 14. Extend ECB Exchange Rates

Combine June–July exchange rates with the historical rate from May 30, 2025.

In [ ]:
df_fx_complete = df_fx.unionByName(
    df_fx_may30
)

print("Total FX Rows:", df_fx_complete.count())
df_fx_complete.orderBy("rate_date").show(5)

## 15. Create Daily FX Calendar

Generate a continuous daily calendar from May 30 to July 31, 2025.

The calendar includes weekends and holidays when ECB exchange rates may be unavailable.

In [ ]:
df_calendar = spark.sql("""
    SELECT explode(
        sequence(
            to_date('2025-05-30'),
            to_date('2025-07-31'),
            interval 1 day
        )
    ) AS rate_date
""")

df_calendar.orderBy("rate_date").show(5)

print("Calendar Rows:", df_calendar.count())

## 16. Join FX Rates to Daily Calendar

Join the complete calendar with ECB exchange rates.

Use a left join to preserve all dates, including weekends and holidays without published exchange rates.

In [ ]:
df_fx_calendar = df_calendar.join(
    df_fx_complete,
    ["rate_date"],
    "left"
)

In [ ]:
df_fx_calendar.orderBy("rate_date").show(7)

print("Rows:", df_fx_calendar.count())

## 17. Fill Missing FX Rates

Use a Spark Window function to forward-fill missing exchange rates with the last available ECB rate.

This handles weekends and holidays when no new exchange rate is published.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last

w_fx = (
    Window
    .orderBy("rate_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

In [ ]:
df_fx_filled = df_fx_calendar.withColumn(
    "usd_per_eur_filled",
    last("usd_per_eur", ignorenulls=True).over(w_fx)
)

In [ ]:
df_fx_filled.orderBy("rate_date").show(7)

In [ ]:
df_fx_filled.filter(
    col("usd_per_eur_filled").isNull()
).count()

## 18. Join Daily Taxi Fact with FX Rates

Join daily taxi metrics with the prepared ECB exchange rates.

Use a left join to preserve all taxi date-zone records.

In [ ]:
df_taxi_with_fx = df_fact_taxi_daily_v2.join(
    df_fx_filled,
    df_fact_taxi_daily_v2.trip_date == df_fx_filled.rate_date,
    "left"
)

In [ ]:
print("Rows after JOIN:", df_taxi_with_fx.count())

In [ ]:
df_taxi_with_fx.filter(
    col("usd_per_eur_filled").isNull()
).count()

## 19. Convert Taxi Revenue from USD to EUR

Convert total revenue and positive payment revenue using the prepared daily ECB exchange rates.

In [ ]:
df_taxi_with_fx = df_taxi_with_fx.withColumn(
    "total_amount_eur",
    df_taxi_with_fx.total_amount_usd /df_taxi_with_fx.usd_per_eur_filled

)

In [ ]:
df_taxi_with_fx =df_taxi_with_fx.withColumn(
    "positive_charged_amount_eur",
    df_taxi_with_fx.positive_charged_amount_usd/df_taxi_with_fx.usd_per_eur_filled
)

## 20. Round Financial Amounts

Round converted EUR amounts to two decimal places before preparing the final Gold Fact table.

In [ ]:
from pyspark.sql.functions import round, col

df_taxi_final = (
    df_taxi_with_fx
    .withColumn(
        "total_amount_eur",
        round(col("total_amount_eur"), 2)
    )
    .withColumn(
        "positive_charged_amount_eur",
        round(col("positive_charged_amount_eur"), 2)
    )
)

## 21. Prepare Financial Columns for Gold

Convert USD and EUR revenue columns to DecimalType(18,2)
to store financial amounts with two decimal places.

In [ ]:
from pyspark.sql.types import DecimalType

df_taxi_gold = (
    df_taxi_final
    .withColumn(
        "total_amount_usd",
        col("total_amount_usd").cast(DecimalType(18,2))
    )
    .withColumn(
        "positive_charged_amount_usd",
        col("positive_charged_amount_usd").cast(DecimalType(18,2))
    )
    .withColumn(
        "total_amount_eur",
        col("total_amount_eur").cast(DecimalType(18,2))
    )
    .withColumn(
        "positive_charged_amount_eur",
        col("positive_charged_amount_eur").cast(DecimalType(18,2))
    )
)

In [ ]:
df_taxi_gold.select(
    "total_amount_usd",
    "positive_charged_amount_usd",
    "total_amount_eur",
    "positive_charged_amount_eur"
).printSchema()

## 22. Validate Gold Fact

Validate total rows, trip counts, missing values, and duplicate date-zone keys before saving the Gold Fact table.

In [ ]:
from pyspark.sql.functions import sum as spark_sum
print("Gold rows:", df_taxi_gold.count())

df_taxi_gold.agg(
    spark_sum("daily_trips").alias("total_trips")
).show()

In [ ]:
df_taxi_gold.filter(
    col("total_amount_eur").isNull()
).count()

In [ ]:
df_taxi_gold.filter(
    col("trip_date").isNull() |
    col("PULocationID").isNull() |
    col("daily_trips").isNull() |
    col("total_amount_usd").isNull() |
    col("positive_charged_amount_usd").isNull() |
    col("positive_charged_amount_eur").isNull()
).count()

In [ ]:
df_duplicates =(
    df_taxi_gold
    .groupBy("trip_date", "PULocationID")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate keys:", df_duplicates.count())

## 23. Save Gold Taxi Daily Fact

Save the daily taxi fact with USD/EUR revenue and prepared exchange rate columns.

This table is an intermediate Gold output before removing FX helper columns.

In [ ]:
df_taxi_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_fact_taxi_daily")

In [ ]:
df_saved = spark.table("dbo.gold_fact_taxi_daily")

print("Saved rows:", df_saved.count())

In [ ]:
df_saved.printSchema()

## 24. Prepare Final Gold Fact

Remove temporary FX columns and keep the final taxi metrics for the analytical model.

In [ ]:
df_fact_clean = df_saved.drop(
    "rate_date",
    "usd_per_eur",
    "usd_per_eur_filled"
)

In [ ]:
df_fact_clean.printSchema()
print("Rows:", df_fact_clean.count())

## 25. Save Final Gold Taxi Daily Fact

Save the final daily taxi fact as a Delta table in the Lakehouse.

In [ ]:
df_fact_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_fact_taxi_daily_ready")

## 26. Validate Final Gold Output

Verify the saved Gold Fact table and its dimensions.

In [ ]:
df_check = spark.table("dbo.gold_fact_taxi_daily_ready")

print("Rows:", df_check.count())
print("Columns:", len(df_check.columns))